In [1]:
import pandas as pd
import numpy as np
import zipfile

zip_path = "preprocessed_data1.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    with z.open("preprocessed_data1.csv") as f:
        prep_df = pd.read_csv(f)

print(prep_df.shape)

(39119, 12)


In [2]:
import re

def count_killers(names_text: str) -> int:
    text = names_text.replace('\n', ' ').replace('\r', ' ').strip()

    people = re.split(r'\s*,\s*(?=[^,]*?\s*-\s*ст\.)', text)

    count = 0
    for person in people:
        # print(person)
        if '105' in person:
            count += 1
    # print("end")
    return count

In [3]:
prep_df['killer_count'] = prep_df['names'].apply(count_killers)

In [4]:
prep_df1 = prep_df[prep_df['preamble'].notna() & prep_df['description'].notna() & prep_df['sentence'].notna()]
prep_df1.shape

(39119, 13)

In [6]:
df_3000 = pd.read_csv('3k_1.csv')

In [7]:
df_unlabeled = prep_df1[~prep_df1['id'].isin(df_3000['id'])].copy()
df_unlabeled.shape

(36119, 13)

In [8]:
df_unlabeled = df_unlabeled[df_unlabeled['killer_count'] == 1]
df_unlabeled.shape

(34765, 13)

In [9]:
import re
import pymorphy3
from nltk.corpus import stopwords
import nltk
from tqdm import tqdm

nltk.download("stopwords")
custom_stopwords = set([
    "в", "мм", "гггг", "дд", "ук", "рф", "ст", "д", "и", "ч", "ходе", "т", "л"
])

all_stopwords = set(stopwords.words("russian")).union(custom_stopwords)
morph = pymorphy3.MorphAnalyzer()

def lemmatize_word(word):
    return morph.parse(word)[0].normal_form

def preprocess_text(text):
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = text.lower()
    return text

tqdm.pandas()
df_unlabeled["tfidf_description_prep_no_lemma"] = df_unlabeled["description"].astype(str).progress_apply(preprocess_text)
df_unlabeled["tfidf_preamble_prep_no_lemma"] = df_unlabeled["preamble"].astype(str).progress_apply(preprocess_text)
df_unlabeled["tfidf_sentence_prep_no_lemma"] = df_unlabeled["sentence"].astype(str).progress_apply(preprocess_text)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Zephyrus\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
100%|██████████| 34765/34765 [00:04<00:00, 7090.33it/s]


In [10]:
import re
import pymorphy3
from nltk.corpus import stopwords
import nltk
from tqdm import tqdm
from functools import lru_cache

nltk.download("stopwords")
custom_stopwords = set([
    "в", "мм", "гггг", "дд", "ук", "рф", "ст", "д", "и", "ч", "ходе", "т", "л"
])

all_stopwords = set(stopwords.words("russian")).union(custom_stopwords)
morph = pymorphy3.MorphAnalyzer()

@lru_cache(maxsize=100000)
def lemmatize_word(word):
    return morph.parse(word)[0].normal_form

def preprocess_text(text):
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = text.lower()
    words = text.split()
    
    lemmas = [lemmatize_word(word) for word in words if word not in all_stopwords]
    return " ".join(lemmas)

tqdm.pandas()
df_unlabeled["tfidf_description_prep"] = df_unlabeled["description"].astype(str).progress_apply(preprocess_text)
df_unlabeled["tfidf_preamble_prep"] = df_unlabeled["preamble"].astype(str).progress_apply(preprocess_text)
df_unlabeled["tfidf_sentence_prep"] = df_unlabeled["sentence"].astype(str).progress_apply(preprocess_text)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Zephyrus\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
100%|██████████| 34765/34765 [00:17<00:00, 2016.37it/s]


In [10]:
# X_unlabeled = vectorizer.transform(df_unlabeled["tfidf_preamble_prep_no_lemma"] + df_unlabeled["tfidf_sentence_prep_no_lemma"])
# df_unlabeled['gender_accused'] = model.predict(X_unlabeled)

In [11]:
# df_unlabeled['gender_accused'].value_counts()

In [12]:
# import joblib

# model_method = joblib.load("saved_models/model_predicted_method_catboost_tfidf_lemma_nobal.joblib")
# vectorizer_method = joblib.load("saved_models/vectorizer_predicted_method_lemma.joblib")

In [11]:
from scipy.sparse import issparse
import joblib

def predict_and_save_to_column(df, text, col, model_path, mlb_path, vectorizer_path):
    model = joblib.load(model_path)
    if mlb_path is not None:
        mlb = joblib.load(mlb_path)
        print(mlb.classes_)
    vectorizer_method = joblib.load(vectorizer_path)

    X = vectorizer_method.transform(text)
    X_input = X.copy()
    # X_input = X.toarray() if issparse(X) else X

    if mlb_path is not None:
        y_pred = model.predict(X_input)
        y_pred_labels = mlb.inverse_transform(y_pred)
        df[col] = [', '.join(labels) if labels else 'другое' for labels in y_pred_labels]
    else:
        df[col] = model.predict(X_input)
    return df

In [14]:
df_unlabeled = predict_and_save_to_column(df=df_unlabeled, 
                                          text=df_unlabeled["tfidf_preamble_prep_no_lemma"] + df_unlabeled["tfidf_sentence_prep_no_lemma"], 
                                          col="gender_accused", 
                                          model_path="saved_models/model_predicted_gender_catboost_tfidf_nolem_nobal.joblib", 
                                          mlb_path=None,
                                          vectorizer_path="saved_models/vectorizer_predicted_gender_nolem.joblib")
df_unlabeled['gender_accused'].value_counts()

gender_accused
мужчина    29232
женщина     5533
Name: count, dtype: int64

In [15]:
df_unlabeled = predict_and_save_to_column(df=df_unlabeled, 
                                          text=df_unlabeled["tfidf_preamble_prep"] + df_unlabeled["tfidf_sentence_prep"], 
                                          col="prior_convictions", 
                                          model_path="saved_models/model_predicted_prior_convictions_catboost_tfidf_lemma_bal.joblib", 
                                          mlb_path=None,
                                          vectorizer_path="saved_models/vectorizer_predicted_prior_convictions_lemma.joblib")
df_unlabeled['prior_convictions'].value_counts()

prior_convictions
нет    27173
да      7592
Name: count, dtype: int64

In [16]:
df_unlabeled = predict_and_save_to_column(df=df_unlabeled,
                                          text=df_unlabeled['tfidf_description_prep_no_lemma'],
                                          col="alcohol",
                                          model_path="saved_models/model_predicted_alcohol_random_forest_tfidf_nolem_bal.joblib",
                                          mlb_path=None,
                                          vectorizer_path="saved_models/vectorizer_predicted_alcohol_nolem.joblib")
df_unlabeled['alcohol'].value_counts()

alcohol
да     31288
нет     3477
Name: count, dtype: int64

In [17]:
df_unlabeled = predict_and_save_to_column(df=df_unlabeled,
                                          text=df_unlabeled['tfidf_description_prep'],
                                          col="precrime_argument",
                                          model_path="saved_models/model_predicted_precrime_argument_random_forest_tfidf_lemma_bal.joblib",
                                          mlb_path=None,
                                          vectorizer_path="saved_models/vectorizer_predicted_precrime_argument_lemma.joblib")
df_unlabeled['precrime_argument'].value_counts()

precrime_argument
была    34666
нет        99
Name: count, dtype: int64

In [18]:
df_unlabeled = predict_and_save_to_column(df=df_unlabeled,
                                          text=df_unlabeled['tfidf_description_prep_no_lemma'],
                                          col="has_woman_victim",
                                          model_path="saved_models/model_predicted_has_woman_victim_random_forest_tfidf_nolem_nobal.joblib",
                                          mlb_path=None,
                                          vectorizer_path="saved_models/vectorizer_predicted_has_woman_victim_nolem.joblib")
df_unlabeled['has_woman_victim'].value_counts()

has_woman_victim
0    24037
1    10728
Name: count, dtype: int64

In [19]:
df_unlabeled = predict_and_save_to_column(df=df_unlabeled,
                                          text=df_unlabeled['tfidf_description_prep_no_lemma'],
                                          col="has_man_victim",
                                          model_path="saved_models/model_predicted_has_man_victim_random_forest_tfidf_nolem_bal.joblib",
                                          mlb_path=None,
                                          vectorizer_path="saved_models/vectorizer_predicted_has_man_victim_nolem.joblib")
df_unlabeled['has_man_victim'].value_counts()

has_man_victim
1.0    27506
0.0     7259
Name: count, dtype: int64

In [20]:
df_unlabeled = predict_and_save_to_column(df=df_unlabeled,
                                          text=df_unlabeled['tfidf_description_prep'],
                                          col="method",
                                          model_path="saved_models/model_predicted_method_catboost_tfidf_lemma_nobal.joblib",
                                          mlb_path="saved_models/mlb_predicted_method.joblib",
                                          vectorizer_path="saved_models/vectorizer_predicted_method_lemma.joblib")
df_unlabeled['method'].value_counts()

['другое' 'огнестрельное оружие' 'тяжелые предметы' 'удушение'
 'физическое воздействие' 'холодное оружие']


method
холодное оружие                                                            17395
физическое воздействие, холодное оружие                                     7803
физическое воздействие                                                      1914
огнестрельное оружие                                                        1675
тяжелые предметы, физическое воздействие                                    1380
удушение, физическое воздействие                                            1116
другое                                                                      1046
удушение                                                                     615
тяжелые предметы                                                             591
тяжелые предметы, физическое воздействие, холодное оружие                    363
тяжелые предметы, холодное оружие                                            202
огнестрельное оружие, физическое воздействие                                 162
удушение, физическое 

In [21]:
df_unlabeled = predict_and_save_to_column(df=df_unlabeled,
                                          text=df_unlabeled['tfidf_description_prep'],
                                          col="motive",
                                          model_path="saved_models/model_predicted_motive_catboost_tfidf_lemma_nobal.joblib",
                                          mlb_path="saved_models/mlb_predicted_motive.joblib",
                                          vectorizer_path="saved_models/vectorizer_predicted_motive_lemma.joblib")
df_unlabeled['motive'].value_counts()

['алкоголь/наркотики' 'аморальность' 'корысть' 'личная неприязнь' 'месть'
 'насилие' 'прочее' 'ревность']


motive
личная неприязнь              32636
корысть, личная неприязнь       916
корысть                         553
другое                          426
личная неприязнь, ревность      164
прочее                           42
ревность                         13
личная неприязнь, прочее         10
корысть, прочее                   5
Name: count, dtype: int64

In [24]:
df_unlabeled['motive'] = df_unlabeled['motive'].replace('прочее', 'другое')
df_unlabeled['motive'] = df_unlabeled['motive'].replace('корысть, прочее', 'корысть, другое')
df_unlabeled['motive'] = df_unlabeled['motive'].replace('личная неприязнь, прочее', 'личная неприязнь, другое')
df_unlabeled['motive'].value_counts()

motive
личная неприязнь              32636
корысть, личная неприязнь       916
корысть                         553
другое                          468
личная неприязнь, ревность      164
ревность                         13
личная неприязнь, другое         10
корысть, другое                   5
Name: count, dtype: int64

In [23]:
df_unlabeled = predict_and_save_to_column(df=df_unlabeled,
                                          text=df_unlabeled['tfidf_description_prep'],
                                          col="location",
                                          model_path="saved_models/model_predicted_location_catboost_tfidf_lemma_nobal.joblib",
                                          mlb_path="saved_models/mlb_predicted_location.joblib",
                                          vectorizer_path="saved_models/vectorizer_predicted_location_lemma.joblib")
df_unlabeled['location'].value_counts()

['другое' 'жилое помещение' 'общественное место' 'общественный транспорт'
 'подъезд и прилегающие зоны' 'рабочая зона' 'уличное пространство']


location
жилое помещение                                                         25987
уличное пространство                                                     4073
жилое помещение, уличное пространство                                    1424
другое                                                                   1208
жилое помещение, подъезд и прилегающие зоны                               444
общественное место, уличное пространство                                  385
подъезд и прилегающие зоны                                                337
общественное место                                                        321
жилое помещение, общественное место                                       145
подъезд и прилегающие зоны, уличное пространство                          143
рабочая зона                                                              114
рабочая зона, уличное пространство                                         86
жилое помещение, рабочая зона                          

In [12]:
import pandas as pd

prep_df2 = pd.read_csv('prep_llm.csv')
prep_df2 = prep_df2[prep_df2['predicted_prison_term'].isna()]
prep_df2.shape

(6995, 18)

In [18]:
df_unlabeled1 = df_unlabeled[df_unlabeled['id'].isin(prep_df2['id'])]
df_unlabeled1.shape

(6995, 19)

In [19]:
df3_2 = pd.read_csv('prep_df3.2.csv')
df3_2 = df3_2[df3_2['predicted_prison_term'].notna()]
df3_2.shape

(583, 18)

In [20]:
df1 = pd.read_csv('prep_df3.csv')
df1 = df1[df1['predicted_prison_term'].notna()]
df1.shape

(467, 18)

In [21]:
df_unlabeled1 = df_unlabeled1[
    ~df_unlabeled1['id'].isin(df3_2['id']) & 
    ~df_unlabeled1['id'].isin(df1['id'])
].copy()

df_unlabeled1.shape

(5945, 19)

In [22]:
df_unlabeled1 = predict_and_save_to_column(df=df_unlabeled1,
                                          text=df_unlabeled1["tfidf_preamble_prep"] + df_unlabeled1["tfidf_sentence_prep"],
                                          col="prison_term",
                                          model_path="saved_models/model_predicted_prison_term_catboost_tfidf_lemma_.joblib",
                                          mlb_path=None,
                                          vectorizer_path="saved_models/vectorizer_predicted_prison_term_lemma.joblib")
df_unlabeled1['prison_term'].value_counts()

prison_term
7.986857     1
9.361625     1
11.209768    1
10.302104    1
4.839561     1
            ..
7.979150     1
8.023633     1
9.779365     1
8.613736     1
7.699321     1
Name: count, Length: 5945, dtype: int64

In [31]:
sum(df_unlabeled1['prison_term'].isna())

0

In [23]:
df_unlabeled1['prison_term'].min(), df_unlabeled1['prison_term'].max()

(1.002754959411746, 446.3285603046516)

In [24]:
df_unlabeled1.loc[df_unlabeled1['prison_term'] >= 100, 'prison_term'] = 100

In [25]:
df_unlabeled1['prison_term'].min(), df_unlabeled1['prison_term'].max()

(1.002754959411746, 100.0)

In [26]:
df_unlabeled1.columns

Index(['id', 'region', 'entryDate', 'names', 'judge', 'decision', 'accused',
       'articles', 'link_text', 'preamble', 'description', 'sentence',
       'killer_count', 'tfidf_description_prep_no_lemma',
       'tfidf_preamble_prep_no_lemma', 'tfidf_sentence_prep_no_lemma',
       'tfidf_description_prep', 'tfidf_preamble_prep', 'tfidf_sentence_prep',
       'prison_term'],
      dtype='object')

In [27]:
df_unlabeled1 = df_unlabeled1.drop(['tfidf_description_prep_no_lemma', 'tfidf_preamble_prep_no_lemma', 'tfidf_sentence_prep_no_lemma', 'tfidf_description_prep', 'tfidf_preamble_prep', 'tfidf_sentence_prep'], axis=1)

In [28]:
df_unlabeled1.columns

Index(['id', 'region', 'entryDate', 'names', 'judge', 'decision', 'accused',
       'articles', 'link_text', 'preamble', 'description', 'sentence',
       'killer_count', 'prison_term'],
      dtype='object')

In [29]:
df_unlabeled1.to_csv('df_labeled_all_1.csv', index=False)